# How to calibrate the geometry

`python -m anri.calib fit` fits the detector and beam geometry (distance, beam centre, tilts, and the wavelength or the lattice scale) to the peaks of a scanning-3DXRD scan of an indexed sample. Each peak is matched to an orientation of an `anri.index` map and a reflection, and scatters from where the beam crosses that orientation's voxels at the peak's dty and ω: the parallax a single-grain fit (ImageD11's `makemap`, `refine_em`, its silicon notebook) cannot model on a scan. Each orientation's rotation is refitted with the geometry. This notebook runs the same stage on your data, with the plots that help you choose its options before you run it and judge the result after.

| Section | Options | What the plots show |
|---|---|---|
| 1. The map | `--min-occ` | how much of the map's occupancy each cut leaves out, voxel by voxel |
| 2. Matching | `--tols`, `--lattice` | how far each peak's (h, k, l) is from integers at the starting geometry: real matches against chance |
| 3. The fit | `--free`, `--iter`, `--clip` | each round's matched peaks, residuals and parameters (the stage's log) |
| 4. Residuals | `--clip` | the residuals against ω, the azimuth on the detector and dty, and their distribution |
| 5. Strain left | `--min-peaks` | each orientation's strain with its lattice free and the geometry fixed: the calibration's precision |
| 6. Files | | the ImageD11 parameter files (`python -m anri.calib pars`) and the command that remakes the calibration |

Where a choice is cheap to redo, a slider redraws the plot, and **the value on the slider is what the later cells use**. The stage is the command line's own function (`anri.calib.stages.stage_fit`), so the file written here is the one `python -m anri.calib fit` writes, with the same provenance (`python -m anri.calib show <file>`). The last cell prints the command that remakes it.

What you need: an indexed map (the fit stage of `python -m anri.index`, or of the [indexing parameters](indexing_parameters.ipynb) guide), indexed with a geometry good enough to index, and ImageD11's peaks table of the same scan (the DataSet's `pksfile`, from its segmentation). Wedge, chi and y0 stay as they were indexed. A calibrant (known lattice, e.g. a silicon or CeO2 crystal) frees the wavelength; any other sample (known wavelength) frees the lattice scale instead: the two are degenerate.

Run it top to bottom; after moving a slider, re-run the cells below it. Figures are interactive (ipympl, `%matplotlib widget`). It needs `ipywidgets`, `ipympl` and `matplotlib`.

With no map given, it renders the 316L [phantom](../tutorials/phantom.ipynb) as in the [indexing tutorial](../tutorials/indexing.ipynb), writes its parameters with a deliberately wrong geometry, segments it into a peaks table and indexes it with the indexer's defaults (several minutes on a laptop CPU; it needs ImageD11), so it runs out of the box and the result can be checked against the true geometry.

In [ ]:
# Parameters: the options of python -m anri.calib fit, at its defaults. The sliders below start from these values.

# the indexed map: an anri.index fit stage file; None renders a phantom with a wrong geometry and indexes it
fit_file = None
peaks = None  # --peaks: ImageD11 peaks table (None: the DataSet's pksfile)
lattice = None  # --lattice: reference (a, b, c, alpha, beta, gamma), e.g. a calibrant's certified cell (None: the phase's)

# 1. the map
min_occ = 0.01  # --min-occ: leave out occupancies below this x the largest
# 2. matching
tols = (0.1, 0.05, 0.025, 0.025)  # --tols: (h, k, l) matching tolerance of each round
# 3. the fit
free = ("distance", "y_center", "z_center", "tilt_x", "tilt_y", "tilt_z", "wavelength")  # --free: "lattice"
# instead of "wavelength" for a sample that is not a calibrant
n_iter = 10  # --iter: Gauss-Newton iterations per round
clip = 4.0  # --clip: leave out peaks beyond this many (robust) sigma
# 5. strain left
min_peaks = 20  # --min-peaks: free the lattice of orientations with at least this many peaks

n_cpu = 4  # --n-cpu
out = None  # --out (None: the map's name with _calib, beside it)
pars_folder = None  # where python -m anri.calib pars writes (None: calib_pars beside the calibration)

In [ ]:
import anri.utils

anri.utils.setup(n_cpu=n_cpu)  # before any JAX computation

%matplotlib widget

import os
import shlex
import tempfile
import time

import h5py
import ipywidgets as widgets
import numpy as np
from IPython.display import display
from matplotlib import pyplot as plt

import anri.calib as cb
import anri.index as ix
import anri.io

T0 = time.time()
NOTEBOOK = "calibration.ipynb"  # recorded as the command in each stage file's provenance
true_pars = None  # the phantom's true geometry, to check the result against


def log(msg):
    print(f"[{time.time() - T0:7.1f} s] {msg}", flush=True)


def figure(name, **kwargs):
    # a figure shown where display() puts it (under its sliders), not at the end of the cell; re-running the cell
    # replaces it
    plt.close(name)
    with plt.ioff():
        fig = plt.figure(name, layout="constrained", **kwargs)
    fig.canvas.header_visible = False
    return fig


def interact(fig, update, **controls):
    # sliders that call a plot's update, which redraws it and returns a summary; their values are in .kwargs
    w = widgets.interactive(lambda **kw: print(update(**kw)), **controls)
    display(w, fig.canvas)
    return w


def slider(value, lo, hi, step, name, log_scale=False):
    kind = widgets.FloatLogSlider if log_scale else widgets.FloatSlider
    extra = {"base": 10} if log_scale else {"readout_format": ".3g"}
    return kind(value=value, min=lo, max=hi, step=step, description=name, continuous_update=False, **extra)

## The phantom

Without a map, the 316L phantom of the [indexing tutorial](../tutorials/indexing.ipynb) is rendered into a temporary folder as an ImageD11 dataset, with its true geometry. Its `pars.json` gets a wrong one, as an experiment's first guess would be: the distance 0.3% long, the wavelength 0.2% long, the beam centre 1.5 and 1 pixels off, the tilts 1 to 2 mrad off. ImageD11 segments the pixels into a peaks table, and the indexer's stages, at their defaults, index the scan with the wrong geometry. The phantom's cells are misoriented by a few tenths of a degree and it has no strain, so it behaves like a calibrant whose grains are a little deformed.

In [ ]:
if fit_file is None:
    from ImageD11.sinograms.tensor_map import TensorMap

    truth = TensorMap.from_h5(os.path.join("..", "..", "..", "tests", "data", "phantoms", "am316l", "am316l_tmap.h5"))
    a = truth.phases[0].lattice_parameters[0]
    wl = 0.2843
    true_pars = {
        "y_center": 1023.5, "y_size": 75.0, "tilt_y": 1e-3, "z_center": 1023.5, "z_size": 75.0, "tilt_z": -2e-3,
        "tilt_x": 0.0, "distance": 150e3, "o11": -1, "o12": 0, "o21": 0, "o22": -1, "wavelength": wl, "wedge": 0.0,
        "chi": 0.0, "omegasign": 1.0, "t_x": 0.0, "t_y": 0.0, "t_z": 0.0,
    }
    y0_true = 0.3
    geom_r = anri.io.imaged11.geom_from_pars(true_pars, y0_true, wl * 2e-4 / 2.355, 5e-5, 5e-5, sig_beam=1.4 / 2.355,
                                             voxel_size=0.5, sig_psf=0.5)
    entries = anri.io.imaged11.entries_from_tensormap(truth)
    entries["density"] = np.full(len(entries["pos"]), 30.0)  # counts scale
    rings8 = ix.data.ring_table(np.array([a, a, a, 90.0, 90.0, 90.0]), 225, wl, 8)
    omega_r, dty_r = anri.io.imaged11.motor_grid((0.0, 180.0), 0.1, (y0_true - 30.0, y0_true + 30.0), 1.0)
    root, sample, dataset = tempfile.mkdtemp(), "phantom", "am316l"
    sparse = os.path.join(root, "phantom_sparse.h5")
    t0 = time.time()
    anri.io.imaged11.simulate_sparse(sparse, entries, rings8["hkls"], np.ones(len(rings8["hkls"])), geom_r, omega_r,
                                     dty_r, (2048, 2048), max_frames=61)
    log(f"phantom rendered in {time.time() - t0:.0f} s into {root}")

    # the geometry the scan is indexed with, and the calibration starts from
    start = dict(true_pars, distance=150.45e3, wavelength=wl * 1.002, y_center=1025.0, z_center=1022.5, tilt_x=1e-3,
                 tilt_y=0.0, tilt_z=0.0)
    cell_r = {"cell__a": a, "cell__b": a, "cell__c": a, "cell_alpha": 90.0, "cell_beta": 90.0, "cell_gamma": 90.0,
              "cell_lattice_[P,A,B,C,I,F,R]": 225}
    pars_r = anri.io.imaged11.write_pars(os.path.join(root, "pars"), start, {"316L": cell_r})
    dsfile = anri.io.imaged11.write_dataset(sparse, root, sample, dataset, y0=y0_true, parfile=pars_r)
    peaks = anri.io.imaged11.write_peaks_table(dsfile, nproc=n_cpu)  # ImageD11's segmentation into 2D and 4D peaks
    log(f"peaks table: {peaks}")

    stem = os.path.join(root, f"{sample}_{dataset}")
    histogram_file = ix.stages.stage_histogram(root, sample, dataset, f"{stem}_histogram.h5", command=NOTEBOOK, log=log)
    search_file = ix.stages.stage_search(histogram_file, command=NOTEBOOK, log=log)
    filter_file = ix.stages.stage_filter(search_file, command=NOTEBOOK, log=log)
    fit_file = ix.stages.stage_fit(filter_file, command=NOTEBOOK, log=log)

## 1. The map: `--min-occ`

Each voxel of the map holds up to `cand` candidate orientations (64 by default), most of them with almost no occupancy. A peak is matched to an orientation that a voxel along its beam path holds, so weak candidates give more chances to match a peak to the wrong orientation by chance; `min_occ` leaves out every occupancy below that fraction of the largest in the map.

The histogram of all occupancies should show the occupied orientations well apart from a floor of near-zero candidates: put the cut in the gap. On the right, the share of each voxel's occupancy that is kept: it should stay near 1 inside the grains. A voxel whose occupancy is all left out cannot place any peak, and peaks from it will not match.

In [ ]:
with h5py.File(fit_file, "r") as h:
    f_map = h["f"][()]
NR = round(np.sqrt(len(f_map)))
f_max, f_tot = f_map.max(), f_map.sum(1)


def plot_min_occ(fig):
    ax = fig.subplots(1, 2, width_ratios=(1.6, 1))
    ax[0].hist(np.log10(f_map[f_map > 0] / f_max), bins=120, log=True, color="tab:blue")
    line = ax[0].axvline(np.log10(min_occ), color="tab:red")
    ax[0].set_xlabel("log10 occupancy / largest")
    ax[0].set_ylabel("(voxel, candidate) pairs")
    im = ax[1].imshow(np.full((NR, NR), np.nan), origin="lower", cmap="viridis", vmin=0, vmax=1)
    ax[1].set_facecolor("0.85")
    ax[1].set_title("share of each voxel's occupancy kept")
    ax[1].set_xticks([]), ax[1].set_yticks([])
    fig.colorbar(im, ax=ax[1], shrink=0.8)

    def update(min_occ):
        kept = np.where(f_map > min_occ * f_max, f_map, 0.0)
        line.set_xdata([np.log10(min_occ)] * 2)
        im.set_data(np.where(f_tot > 0, kept.sum(1) / np.maximum(f_tot, 1e-30), np.nan).reshape(NR, NR))
        fig.canvas.draw_idle()
        return (f"{(kept > 0).sum()} of {(f_map > 0).sum()} (voxel, orientation) pairs kept on "
                f"{np.any(kept > 0, 1).sum()} voxels, holding {kept.sum() / f_map.sum() * 100:.1f}% of the occupancy")

    return update


fig = figure("min_occ", figsize=(13, 4.2))
w_occ = interact(fig, plot_min_occ(fig), min_occ=slider(min_occ, -5, 0, 0.05, "min_occ", log_scale=True))

## 2. Matching: `--tols`, `--lattice`

The stage reads the peaks (ImageD11's 2D peaks merged per 4D label and dty row, `anri.io.imaged11.read_peaks`) and the map (`anri.calib.stages.load_inputs`), and matches every peak at the starting geometry (`anri.calib.fit.match`): the beam's centre line at the peak's dty and ω is sampled at every voxel it crosses, each orientation held there gives the peak's (h, k, l), and the match is the orientation and point whose (h, k, l) is nearest to integers. The fit uses the peaks whose distance from integers is within the tolerance of each round, and after each round matches again with the new geometry and orientations, so the tolerances can shrink from round to round.

What to look for:

- **Real matches pile up near 0**, above a broad floor of chance matches (the nearest of many orientations and points along the ray is always somewhat near integers). The first tolerance should take in the whole pile at the starting geometry, and not much of the floor. A wrong geometry, or orientations off by up to half the indexing grid step, widen the pile: the larger (h, k, l), the wider, as the scatter against intensity shows for weak (often high-angle) peaks.
- **No pile at all**: the geometry is too far off, the map's lattice is wrong (`--lattice`), or the peaks are not this scan's.
- The later tolerances (the grey lines) apply after the geometry has been fitted once; section 4 shows the distances again with the calibrated geometry.

The slider sets the first round's tolerance; the later rounds keep theirs from the parameters.

In [ ]:
d = cb.stages.load_inputs(fit_file, peaks, lattice, w_occ.kwargs["min_occ"])
pk = d["peaks"]
t0 = time.time()
m0 = cb.fit.match(pk, d["ubi"], d["grid_o"], d["grid_w"], d["step"], d["pars"])
seen = m0["weight"] > 0  # peaks whose beam path crosses a voxel of the map
log(f"{len(pk['sc'])} peaks from {d['peaks_file']} matched in {time.time() - t0:.0f} s; {seen.sum()} cross the map")
DIST_BINS = np.linspace(0.0, 0.87, 175)  # (h, k, l) is at most sqrt(3) / 2 from integers


def plot_match(fig, err, intensity):
    ax = fig.subplots(1, 2)
    ax[0].hist(err, DIST_BINS, log=True, color="tab:blue")
    for t in tols[1:]:
        ax[0].axvline(t, color="0.5", ls=":")
    line = ax[0].axvline(tols[0], color="tab:red")
    ax[0].set_xlabel("distance of (h, k, l) from integers")
    ax[0].set_ylabel("peaks")
    sub = np.random.default_rng(0).choice(len(err), min(len(err), 20000), replace=False)
    ax[1].scatter(intensity[sub], err[sub], s=2, alpha=0.3, color="tab:blue")
    hline = ax[1].axhline(tols[0], color="tab:red")
    ax[1].set_xscale("log")
    ax[1].set_xlabel("peak intensity")
    ax[1].set_ylabel("distance from integers")

    def update(tol):
        line.set_xdata([tol, tol])
        hline.set_ydata([tol, tol])
        fig.canvas.draw_idle()
        return f"{np.sum(err < tol)} of {len(err)} peaks ({np.mean(err < tol) * 100:.1f}%) within {tol:g} in the first round"

    return update


fig = figure("match", figsize=(14, 4))
w_tol = interact(fig, plot_match(fig, m0["err"][seen], pk["intensity"][seen]),
                 tol=slider(tols[0], 0.005, 0.4, 0.005, "tol"))

## 3. The fit: `--free`, `--iter`, `--clip`

The stage itself, as `python -m anri.calib fit` runs it. Each round matches the peaks, keeps one peak per (orientation, reflection, branch, dty row), leaves out those further than `clip` robust standard deviations from their prediction, and fits the free parameters and every orientation's rotation by Levenberg-Marquardt (`anri.calib.fit.calibrate`). Then, with the geometry fixed, each orientation with at least `min_peaks` peaks gets its lattice freed (`anri.calib.fit.strain_left`, section 5).

In the log, from round to round: the matched peaks should grow or stay, the standard deviations (sigma) in sc, fc and ω should fall to the peaks' own precision, and the parameters settle within their error bars. A parameter that moves by many of its error bars in the last round has not converged: add a round (`tols`) or iterations (`n_iter`).

In [ ]:
tols_used = (w_tol.kwargs["tol"], *tols[1:])
min_occ_used = w_occ.kwargs["min_occ"]
calib_file = cb.stages.stage_fit(fit_file, out, peaks=peaks, lattice=lattice, free=tuple(free), tols=tols_used,
                                 n_iter=n_iter, clip=clip, min_occ=min_occ_used, min_peaks=min_peaks,
                                 command=NOTEBOOK, log=log)
prov = anri.io.provenance.read_provenance(calib_file)
with h5py.File(calib_file, "r") as h:
    geo = dict(h["geometry"].attrs)
    cov, sig, ubi_fit = h["cov"][()], h["sig"][()], h["ubi"][()]
    E, n_pk = h["strain"][()], h["n_peaks"][()]
    mt = {k: h[f"matched/{k}"][()] for k in h["matched"]}

err_fit = np.sqrt(np.diag(cov))
print(f"{'':12s}{'start':>14s}{'fitted':>14s}{'+-':>10s}{'moved / +-':>12s}" + ("" if true_pars is None else f"{'truth':>14s}{'off / +-':>10s}"))
for k, e in zip(free, err_fit):
    v0, v = prov["resolved"][f"start_{k}"], float(geo["lattice_scale"] if k == "lattice" else geo[k])
    line = f"{k:12s}{v0:14.7g}{v:14.7g}{e:10.2g}{(v - v0) / e:12.1f}"
    if true_pars is not None:
        t = 1.0 if k == "lattice" else true_pars[k]
        line += f"{t:14.7g}{(v - t) / e:10.1f}"
    print(line)

## 4. Residuals: `--clip`

Each matched peak's observed − predicted (sc, fc, ω), in units of the standard deviations the last round measured, against ω, the azimuth on the detector (about the fitted beam centre) and dty. The red lines are medians in bins. With the right geometry they are flat at zero, and what is left is the peaks' noise.

- **A sinusoid against azimuth** (one period around the detector): the beam centre or tilts are not right; with them free, a pattern of higher order is detector distortion (check the DataSet's distortion files).
- **A trend against ω**: something that turns with the sample and is fixed here, the wedge or chi, or a wobble of the rotation axis.
- **A trend against dty**: y0 (fixed here, from the map), or the parallax of large grains placed wrongly by the map.
- **Heavy tails** in the histograms (beyond the Gaussian): wrong matches or overlapping peaks. A lower `clip` or tighter `tols` leave out more of them; the clip lines show where the last round cut.

Below them, the distances of (h, k, l) from integers again, with the calibrated geometry and orientations: the pile of real matches should be much narrower than at the start, and the later tolerances can be set from it.

In [ ]:
r = mt["residual"] / sig  # [P, 3]: sc, fc, omega
azimuth = np.degrees(np.arctan2(mt["sc"] - geo["z_center"], mt["fc"] - geo["y_center"]))
against = {"ω (°)": mt["omega"], "azimuth on the detector (°)": azimuth, "dty": mt["dty"]}
print(f"{len(r)} peaks; sigma (sc, fc, omega): {sig[0]:.3f} px, {sig[1]:.3f} px, {sig[2]:.4f}°; "
      f"rms residual / sigma: {np.sqrt(np.mean(r**2, 0)).round(2)}")

fig = figure("residuals", figsize=(15, 9))
ax = fig.subplots(3, 3, sharey="row")
sub = np.random.default_rng(0).choice(len(r), min(len(r), 20000), replace=False)
for j, (name, x) in enumerate(against.items()):
    edges = np.linspace(x.min(), x.max(), 41)
    b = np.clip(np.digitize(x, edges) - 1, 0, 39)
    for i, comp in enumerate(("sc", "fc", "ω")):
        med = [np.median(r[b == k, i]) if np.any(b == k) else np.nan for k in range(40)]
        ax[i, j].scatter(x[sub], r[sub, i], s=1, alpha=0.2, color="tab:blue")
        ax[i, j].plot((edges[:-1] + edges[1:]) / 2, med, color="tab:red")
        ax[i, j].axhline(0, color="k", lw=0.5)
        ax[i, j].set_ylim(-clip, clip)
        ax[0, j].set_title(f"against {name.split(' (')[0]}")
        ax[i, 0].set_ylabel(f"{comp}: residual / σ")
        ax[2, j].set_xlabel(name)
display(fig.canvas)

fig = figure("residual_hist", figsize=(15, 3.4))
ax = fig.subplots(1, 4)
z = np.linspace(-clip, clip, 200)
for i, comp in enumerate(("sc", "fc", "ω")):
    ax[i].hist(r[:, i], bins=np.linspace(-clip, clip, 81), log=True, color="tab:blue")
    ax[i].plot(z, len(r) * (2 * clip / 80) * np.exp(-z**2 / 2) / np.sqrt(2 * np.pi), color="k", lw=0.8)
    ax[i].set_xlabel(f"{comp}: residual / σ")
    ax[i].set_ylim(0.5, None)
ax[0].set_ylabel("peaks")

pars_fit = {**cb.fit.base_pars(geo, d["pars"]["y0"]), "lattice": float(geo["lattice_scale"])}
m_fit = cb.fit.match(pk, ubi_fit, d["grid_o"], d["grid_w"], d["step"], pars_fit)
ax[3].hist(m0["err"][seen], DIST_BINS, log=True, histtype="step", label="start")
ax[3].hist(m_fit["err"][m_fit["weight"] > 0], DIST_BINS, log=True, histtype="step", label="calibrated")
for t in tols_used:
    ax[3].axvline(t, color="0.5", ls=":")
ax[3].set_xlabel("distance of (h, k, l) from integers")
ax[3].legend()
display(fig.canvas)

## 5. Strain left: `--min-peaks`

With the geometry fixed, each orientation with at least `min_peaks` matched peaks gets its lattice freed (six strains with its rotation), as ImageD11's silicon notebook estimates its precision. On a strain-free calibrant, a right geometry leaves no strain beyond the noise:

- **A deviatoric strain shared by all orientations** (its peak-weighted mean, the black line, away from zero) shows the detector's tilts or centre; a **hydrostatic** one the wavelength or distance against the reference lattice. (With the lattice scale free, the hydrostatic mean is absorbed by the scale.)
- **The scatter between orientations** is the noise of each one's fit, and falls with its number of peaks. Orientations far outside it, with many peaks, are deformed grains, or orientations that took another's peaks.
- `chi2 x` in the log is how much freeing the lattices lowered the χ²: close to 1, the geometry explains the peaks as well as free lattices do.

On a sample that is not strain-free the strain left is the sample's, plus the calibration's error.

In [ ]:
hydro = np.trace(E, axis1=1, axis2=2) / 3
fitted_o = np.isfinite(hydro)
dev = E - hydro[:, None, None] * np.eye(3)
components = {"xx": (0, 0), "yy": (1, 1), "zz": (2, 2), "yz": (1, 2), "xz": (0, 2), "xy": (0, 1)}
fig = figure("strain", figsize=(15, 4))
ax = fig.subplots(1, 2)
if fitted_o.any():
    w = n_pk[fitted_o]
    ax[0].scatter(n_pk[fitted_o], hydro[fitted_o] * 1e5, s=8, color="tab:blue")
    ax[0].axhline(np.average(hydro[fitted_o], weights=w) * 1e5, color="k")
    ax[0].set_title("hydrostatic")
    for c, (i, j) in components.items():
        pts = ax[1].scatter(n_pk[fitted_o], dev[fitted_o, i, j] * 1e5, s=8, label=c)
        ax[1].axhline(np.average(dev[fitted_o, i, j], weights=w) * 1e5, color=pts.get_facecolor()[0], lw=1)
    ax[1].set_title("deviatoric (sample frame)")
    ax[1].legend(ncol=3)
    for a_ in ax:
        a_.set_xscale("log")
        a_.set_xlabel("matched peaks of the orientation")
        a_.set_ylabel("strain (1e-5)")
display(fig.canvas)
print(f"{fitted_o.sum()} of {len(E)} orientations with at least {min_peaks} peaks")
for k, v in prov["results"].items():
    print(f"  {k}: {v}")

## 6. The parameter files, and the command line

`anri.calib.stages.write_pars` writes the calibration as ImageD11 parameter files (`geometry.par`, `<phase>.par` and `pars.json`; the phase's cell scaled by the fitted lattice scale if it was free), as `python -m anri.calib pars` does. Index again with them (`python -m anri.index ... --parfile <pars.json>`, or `parfile` in the [indexing parameters](indexing_parameters.ipynb) guide), or give them to ImageD11.

Then the `python -m anri.calib fit` command that remakes the calibration file with every option it recorded (`anri.calib.stages.run_command`), and how the file was made, all the way back to the dataset (`anri.io.provenance.show`, as `python -m anri.calib show`).

In [ ]:
pars_folder = pars_folder or os.path.join(os.path.dirname(os.path.abspath(calib_file)), "calib_pars")
new_parfile = cb.stages.write_pars(calib_file, pars_folder)
print(f"-> {new_parfile}", end="\n\n")
print(cb.stages.run_command(calib_file))
print(shlex.join(["python", "-m", "anri.calib", "pars", calib_file, pars_folder]), end="\n\n")
print(anri.io.provenance.show(calib_file))